# 01 — Load & Patch Data

**Goal of this notebook:** load the raw dataset, confirm what shape it's in, and fix a real problem we found in it — the delivery-time column has no relationship to anything else in the data.

Output: `data/order_delivery_patched.csv`


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("../data/Order_delivery_raw.csv")
INTERIM_PATH = Path("../data/order_delivery_patched.csv")

pd.set_option("display.max_columns", None)

## 2. Load raw data

In [2]:
df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head()

(100002, 23)


,Order_ID,User_ID,Restaurant_ID,Driver_ID,Item_Name,Quantity,Total_Price,Order_Time,Delivery_Time,Delivery_Duration_Minutes,City,Payment_Method,Order_Status,Driver_Vehicle,Restaurant_Lat,Restaurant_Lon,Customer_Lat,Customer_Lon,Driver_Lat,Driver_Lon,Delivery_Distance_km,Traffic_Level,Driver_Availability
0,1,U3522,358,485,Fried Chicken,3,273.72,16-06-2025 08:32,16-06-2025 09:11,39,Alexandria,Wallet,Delivered,Motorbike,31.195082,29.921931,31.191404,29.904982,31.215658,29.910664,1.666106,High,Offline
1,2,U9214,316,65,Sandwich,3,365.82,03-06-2025 21:27,03-06-2025 22:00,33,Zagazig,Credit Card,Delivered,Motorbike,30.605729,31.503079,30.586047,31.485820,30.580329,31.502380,2.738698,Low,Online
2,3,U7307,357,309,Koshary,3,401.94,01-06-2025 14:48,01-06-2025 15:26,38,Assiut,Cash,In Transit,Car,27.190180,31.177741,27.164869,31.169218,27.162976,31.189458,2.929079,Medium,Online
3,4,U3612,420,32,Sushi,2,221.18,13-06-2025 02:30,13-06-2025 03:22,52,Mansoura,Cash,Delivered,Car,31.041846,31.381229,31.035773,31.380440,31.054690,31.401187,0.677498,Low,Online
4,5,U3492,73,364,Koshary,5,355.55,06-06-2025 09:48,06-06-2025 10:32,44,Mansoura,Wallet,Delivered,Motorbike,31.024141,31.376104,31.026023,31.396881,31.035350,31.389315,1.994769,High,Online


## 3. Quick structural check

No missing values or duplicates expected — this dataset was already clean when we inspected it originally. We check anyway rather than assume.

In [10]:
print("Missing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate Order_ID:", df['Order_ID'].duplicated().sum())

Missing values:
 Order_ID                     0
User_ID                      0
Restaurant_ID                0
Driver_ID                    0
Item_Name                    0
Quantity                     0
Total_Price                  0
Order_Time                   0
Delivery_Time                0
Delivery_Duration_Minutes    0
City                         0
Payment_Method               0
Order_Status                 0
Driver_Vehicle               0
Restaurant_Lat               0
Restaurant_Lon               0
Customer_Lat                 0
Customer_Lon                 0
Driver_Lat                   0
Driver_Lon                   0
Delivery_Distance_km         0
Traffic_Level                0
Driver_Availability          0
dtype: int64

Duplicate rows: 0
Duplicate Order_ID: 0


## 4. The problem: `Delivery_Duration_Minutes` has no signal

Before building anything on top of this target, let's check whether it actually relates to any other column. A quick correlation check settles it.

In [4]:
num_cols = ['Quantity', 'Total_Price', 'Delivery_Distance_km']
print(df[num_cols + ['Delivery_Duration_Minutes']].corr()['Delivery_Duration_Minutes'])

print("\nMean duration by Traffic_Level:")
print(df.groupby('Traffic_Level')['Delivery_Duration_Minutes'].mean())

print("\nMean duration by Driver_Vehicle:")
print(df.groupby('Driver_Vehicle')['Delivery_Duration_Minutes'].mean())

Quantity                     0.001238
Total_Price                 -0.000709
Delivery_Distance_km        -0.001676
Delivery_Duration_Minutes    1.000000
Name: Delivery_Duration_Minutes, dtype: float64

Mean duration by Traffic_Level:
Traffic_Level
High      37.450840
Low       37.582726
Medium    37.486636
Name: Delivery_Duration_Minutes, dtype: float64

Mean duration by Driver_Vehicle:
Driver_Vehicle
Bicycle      37.528344
Car          37.550310
Motorbike    37.481832
Name: Delivery_Duration_Minutes, dtype: float64


All correlations are ~0.001 and group means are nearly identical across categories. This column is uniform random noise (15–60 min), independent of distance, traffic, vehicle — everything.

## 5. Patch the duration column

Formula:
```
duration = prep_time(item) + travel_time(distance, vehicle, traffic) + dispatch_delay(offline driver) + noise
```

- **Prep time**: item-dependent (Sushi/Pizza slower than Salad/Sandwich)
- **Travel time**: distance ÷ effective speed, where effective speed = vehicle base speed × traffic multiplier
- **Dispatch delay**: if the driver was Offline, add a realistic wait
- **Noise**: small random jitter so it's not deterministic

In [5]:
PREP_TIME_BASE = {
    "Salad": 5, "Sandwich": 6, "Burger": 8, "Pizza": 12,
    "Shawarma": 7, "Fried Chicken": 10, "Pasta": 11,
    "Koshary": 9, "Sushi": 14,
}
VEHICLE_SPEED_KMH = {"Bicycle": 14, "Motorbike": 32, "Car": 24}
TRAFFIC_MULTIPLIER = {"Low": 1.0, "Medium": 0.7, "High": 0.45}

rng = np.random.default_rng(42)

prep_mean = df["Item_Name"].map(PREP_TIME_BASE)
prep_time = np.clip(rng.normal(prep_mean, 2.0), 2, None)

base_speed = df["Driver_Vehicle"].map(VEHICLE_SPEED_KMH)
traffic_mult = df["Traffic_Level"].map(TRAFFIC_MULTIPLIER)
effective_speed = base_speed * traffic_mult
travel_time = (df["Delivery_Distance_km"] / effective_speed) * 60

dispatch_delay = np.where(
    df["Driver_Availability"] == "Offline",
    rng.uniform(5, 18, size=len(df)),
    0.0,
)

noise = rng.normal(0, 3.0, size=len(df))

duration = prep_time + travel_time + dispatch_delay + noise
duration = np.clip(duration, 8, 120).round().astype(int)

df["Delivery_Duration_Minutes"] = duration
df["Delivery_Duration_Minutes"].describe()

count    100002.000000
mean         19.556319
std           8.745432
min           8.000000
25%          13.000000
50%          18.000000
75%          24.000000
max          76.000000
Name: Delivery_Duration_Minutes, dtype: float64

## 6. Verify the patch actually created signal

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder

feat_df = df.copy()
for c in ['City','Payment_Method','Order_Status','Driver_Vehicle','Traffic_Level','Driver_Availability','Item_Name']:
    feat_df[c] = LabelEncoder().fit_transform(feat_df[c])

X = feat_df[['Quantity','Total_Price','City','Payment_Method','Order_Status','Driver_Vehicle',
             'Delivery_Distance_km','Traffic_Level','Driver_Availability','Item_Name']]
y = feat_df['Delivery_Duration_Minutes']

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
print("R^2 on held-out data:", r2_score(yte, rf.predict(Xte)))

R^2 on held-out data: 0.8189562993247584


## 7. Save patched data

In [7]:
INTERIM_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(INTERIM_PATH, index=False)
print(f"Saved -> {INTERIM_PATH}")

Saved -> ..\data\order_delivery_patched.csv
